# Venue / Scorekeeper Bias Analysis (Phase 2, Area 4)

**Goal:** Identify venues where shot recording patterns deviate significantly from the league average, suggesting scorekeeper bias rather than genuine home-ice effects.

**Key questions:**
1. Which venues are statistical outliers in shot count or coordinate distribution?
2. Is it scorekeeper bias or real home-ice advantage? (compare away-team metrics at outlier venues vs. elsewhere)
3. How stable is venue bias across seasons?
4. What correction approach is warranted? (categorical feature vs. pre-correction vs. hierarchical model)

**Data source:** `shot_events`, `games`, and `venue_bias_diagnostics` tables in `data/nhl_data.db`

In [ ]:
import os
import sys
import sqlite3

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Add src/ to path — handle CWD being project root or notebooks/
for _candidate in [os.path.join(os.getcwd(), "src"),
                   os.path.join(os.getcwd(), "..", "src")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from database import DATABASE_PATH

sns.set_theme(style="whitegrid")
print(f"Database: {DATABASE_PATH}")
conn = sqlite3.connect(DATABASE_PATH)
conn.row_factory = sqlite3.Row
print("Connected.")

## 1. Data availability check

Verify venue metadata is populated in the `games` table and check how many seasons/venues are available.

In [ ]:
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM games")
total_games = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM games WHERE venue_name IS NOT NULL")
with_venue = cur.fetchone()[0]

cur.execute("SELECT COUNT(DISTINCT venue_name) FROM games WHERE venue_name IS NOT NULL")
distinct_venues = cur.fetchone()[0]

cur.execute("SELECT COUNT(DISTINCT season) FROM games WHERE venue_name IS NOT NULL")
distinct_seasons = cur.fetchone()[0]

cur.execute("""
    SELECT COUNT(*)
    FROM shot_events se
    JOIN games g ON se.game_id = g.game_id
    WHERE g.venue_name IS NOT NULL
""")
shots_with_venue = cur.fetchone()[0]

print(f"Total games:                     {total_games:,}")
print(f"Games with venue_name:           {with_venue:,} ({with_venue/total_games*100:.1f}%)" if total_games else "")
print(f"Distinct venues:                 {distinct_venues}")
print(f"Distinct seasons:                {distinct_seasons}")
print(f"Shot events joinable to venues:  {shots_with_venue:,}")

if with_venue == 0:
    print("\n*** No venue data. Run the scraper to populate games with venue metadata. ***")

## 2. Per-venue shot counts (league-wide view)

Rank venues by total shots recorded per game. Outlier venues may have scorekeepers who count more liberally (especially missed/blocked shots).

In [ ]:
cur.execute("""
    SELECT g.venue_name,
           COUNT(DISTINCT g.game_id) AS games,
           COUNT(se.shot_event_id)   AS total_shots,
           ROUND(CAST(COUNT(se.shot_event_id) AS REAL) / COUNT(DISTINCT g.game_id), 1) AS shots_per_game,
           ROUND(AVG(se.distance_to_goal), 1) AS avg_distance,
           ROUND(CAST(SUM(se.is_goal) AS REAL) / COUNT(se.shot_event_id), 4) AS goal_rate
    FROM games g
    JOIN shot_events se ON g.game_id = se.game_id
    WHERE g.venue_name IS NOT NULL
    GROUP BY g.venue_name
    HAVING games >= 10
    ORDER BY shots_per_game DESC
""")

venue_rows = cur.fetchall()

if venue_rows:
    names = [r[0] for r in venue_rows]
    spg = [r[3] for r in venue_rows]
    avg_d = [r[4] for r in venue_rows]
    games_ct = [r[1] for r in venue_rows]

    league_avg_spg = sum(r[2] for r in venue_rows) / sum(r[1] for r in venue_rows)

    fig, ax = plt.subplots(figsize=(14, max(6, len(names) * 0.3)))
    colors = ["#e74c3c" if abs(s - league_avg_spg) > 5 else "#3498db" for s in spg]
    y_pos = range(len(names))
    ax.barh(y_pos, spg, color=colors, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel("Shots per game")
    ax.set_title("Shots Per Game by Venue (red = >5 from league avg)")
    ax.axvline(x=league_avg_spg, color="black", linestyle="--", linewidth=1,
               label=f"League avg: {league_avg_spg:.1f}")
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    fig.tight_layout()
    plt.show()

    print(f"\nLeague average shots/game: {league_avg_spg:.1f}")
    print(f"\n{'Venue':<35} {'Games':>6} {'Shots/G':>8} {'Avg Dist':>9} {'Goal Rate':>10}")
    print("-" * 72)
    for r in venue_rows[:10]:
        print(f"{r[0]:<35} {r[1]:>6} {r[3]:>8.1f} {r[4]:>9.1f} {r[5]:>10.4f}")
    if len(venue_rows) > 10:
        print(f"  ... and {len(venue_rows) - 10} more venues")
else:
    print("No venue shot data available.")

## 3. Venue bias diagnostics (z-scores)

Populate the `venue_bias_diagnostics` table and inspect outliers. Venues with |z-score| > 2 in shot count or average distance are flagged.

In [ ]:
from database import populate_venue_diagnostics

# Get all seasons with venue data
cur.execute("""
    SELECT DISTINCT g.season FROM games g
    WHERE g.venue_name IS NOT NULL AND g.season IS NOT NULL
    ORDER BY g.season
""")
seasons = [r[0] for r in cur.fetchall()]
print(f"Seasons with venue data: {len(seasons)}")

# Populate diagnostics for each season
for season in seasons:
    populate_venue_diagnostics(conn, str(season))
print("Diagnostics populated.")

# Show flagged venues
cur.execute("""
    SELECT venue_name, season, total_shots, avg_distance,
           shot_count_z_score, distance_z_score, bias_flag
    FROM venue_bias_diagnostics
    WHERE bias_flag = 1
    ORDER BY ABS(shot_count_z_score) DESC
""")
flagged = cur.fetchall()

if flagged:
    print(f"\nFlagged venues (|z| > 2): {len(flagged)} venue-seasons")
    print(f"\n{'Venue':<35} {'Season':<10} {'Shots':>8} {'Avg Dist':>9} {'Shot Z':>8} {'Dist Z':>8}")
    print("-" * 82)
    for r in flagged:
        sc_z = f"{r[4]:.2f}" if r[4] is not None else "N/A"
        d_z = f"{r[5]:.2f}" if r[5] is not None else "N/A"
        print(f"{r[0]:<35} {str(r[1]):<10} {r[2]:>8,} {r[3]:>9.1f} {sc_z:>8} {d_z:>8}")
else:
    print("\nNo venues flagged as outliers.")

## 4. Z-score scatter plot (shot count vs. distance)

Visualize all venue-seasons in z-score space to identify clusters and outliers.

In [ ]:
cur.execute("""
    SELECT venue_name, season, shot_count_z_score, distance_z_score, bias_flag
    FROM venue_bias_diagnostics
    WHERE shot_count_z_score IS NOT NULL AND distance_z_score IS NOT NULL
""")
diag_rows = cur.fetchall()

if diag_rows:
    names = [r[0] for r in diag_rows]
    sc_z = np.array([r[2] for r in diag_rows])
    d_z = np.array([r[3] for r in diag_rows])
    flags = [r[4] for r in diag_rows]

    fig, ax = plt.subplots(figsize=(10, 8))

    # Normal venues
    normal = [i for i, f in enumerate(flags) if f == 0]
    flagged_idx = [i for i, f in enumerate(flags) if f == 1]

    ax.scatter(sc_z[normal], d_z[normal], alpha=0.4, s=20, color="#3498db", label="Normal")
    ax.scatter(sc_z[flagged_idx], d_z[flagged_idx], alpha=0.8, s=50, color="#e74c3c",
               marker="^", label="Flagged (|z| > 2)")

    # Label flagged venues
    for i in flagged_idx:
        ax.annotate(f"{names[i]}", (sc_z[i], d_z[i]),
                    fontsize=7, alpha=0.8, xytext=(5, 5),
                    textcoords="offset points")

    # Threshold box
    THRESHOLD = 2.0
    ax.axvline(x=THRESHOLD, color="#e74c3c", linestyle=":", alpha=0.5)
    ax.axvline(x=-THRESHOLD, color="#e74c3c", linestyle=":", alpha=0.5)
    ax.axhline(y=THRESHOLD, color="#e74c3c", linestyle=":", alpha=0.5)
    ax.axhline(y=-THRESHOLD, color="#e74c3c", linestyle=":", alpha=0.5)

    ax.set_xlabel("Shot count z-score")
    ax.set_ylabel("Avg distance z-score")
    ax.set_title("Venue Bias: Shot Count vs. Distance Z-Scores")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No diagnostics data available.")

## 5. Home-away asymmetry test

The key diagnostic: is a venue's high/low shot count caused by the scorekeeper, or by the home team's actual playing style? If ALL visiting teams show inflated counts at a venue, it's likely scorekeeper bias. If only the home team shows inflation, it may be real home-ice advantage.

For each flagged venue, compare away teams' shot metrics **at that venue** vs. **the same away teams' metrics elsewhere**.

In [ ]:
# For each flagged venue, compare away team shot counts at that venue
# vs. the same teams' away shot counts at all other venues

cur.execute("""
    SELECT DISTINCT venue_name FROM venue_bias_diagnostics WHERE bias_flag = 1
""")
flagged_venues = [r[0] for r in cur.fetchall()]

if flagged_venues:
    results = []
    for venue in flagged_venues:
        # Away teams' shots per game AT this venue
        cur.execute("""
            SELECT CAST(COUNT(se.shot_event_id) AS REAL) / COUNT(DISTINCT g.game_id)
            FROM shot_events se
            JOIN games g ON se.game_id = g.game_id
            WHERE g.venue_name = ?
              AND se.shooting_team_id = g.away_team_id
        """, (venue,))
        at_venue = cur.fetchone()[0]

        # Same away teams' shots per game at OTHER venues
        cur.execute("""
            SELECT CAST(COUNT(se.shot_event_id) AS REAL) / COUNT(DISTINCT g.game_id)
            FROM shot_events se
            JOIN games g ON se.game_id = g.game_id
            WHERE g.venue_name != ?
              AND se.shooting_team_id = g.away_team_id
              AND g.away_team_id IN (
                  SELECT DISTINCT away_team_id FROM games WHERE venue_name = ?
              )
        """, (venue, venue))
        elsewhere = cur.fetchone()[0]

        diff = at_venue - elsewhere if (at_venue and elsewhere) else None
        results.append((venue, at_venue, elsewhere, diff))

    print(f"{'Venue':<35} {'At Venue':>10} {'Elsewhere':>10} {'Diff':>8}")
    print("-" * 67)
    for venue, at_v, els, diff in results:
        at_s = f"{at_v:.1f}" if at_v else "N/A"
        el_s = f"{els:.1f}" if els else "N/A"
        d_s = f"{diff:+.1f}" if diff is not None else "N/A"
        print(f"{venue:<35} {at_s:>10} {el_s:>10} {d_s:>8}")

    print("\nPositive diff = away teams record MORE shots at this venue than elsewhere")
    print("  -> suggests scorekeeper is counting more liberally")
    print("Negative diff = away teams record FEWER shots at this venue")
    print("  -> could be real defensive home-ice effect or conservative scorekeeper")
else:
    print("No flagged venues to analyze.")

## 6. Coordinate distribution by venue

Do some venues show systematically different x/y coordinate distributions? This can indicate that the scorekeeper is guessing shot locations differently.

In [ ]:
cur.execute("""
    SELECT venue_name, x_coord_mean, x_coord_stddev, y_coord_mean, y_coord_stddev,
           total_shots
    FROM venue_bias_diagnostics
    WHERE x_coord_mean IS NOT NULL
    ORDER BY total_shots DESC
""")
coord_rows = cur.fetchall()

if coord_rows:
    # Plot x_coord stddev vs y_coord stddev — venues with unusual spread stand out
    v_names = [r[0] for r in coord_rows]
    x_std = np.array([r[2] for r in coord_rows])
    y_std = np.array([r[4] for r in coord_rows])
    sizes = np.array([r[5] for r in coord_rows])

    # Normalize sizes for scatter
    size_norm = 20 + 80 * (sizes - sizes.min()) / (sizes.max() - sizes.min() + 1)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(x_std, y_std, s=size_norm, alpha=0.5, color="#3498db")

    # Label outlier coordinate venues (top/bottom 5% in either dimension)
    x_thresh_hi = np.percentile(x_std, 95)
    x_thresh_lo = np.percentile(x_std, 5)
    y_thresh_hi = np.percentile(y_std, 95)
    y_thresh_lo = np.percentile(y_std, 5)

    for i, name in enumerate(v_names):
        if x_std[i] > x_thresh_hi or x_std[i] < x_thresh_lo or \
           y_std[i] > y_thresh_hi or y_std[i] < y_thresh_lo:
            ax.annotate(name, (x_std[i], y_std[i]), fontsize=7, alpha=0.8,
                        xytext=(5, 5), textcoords="offset points")

    ax.set_xlabel("X-coordinate std dev")
    ax.set_ylabel("Y-coordinate std dev")
    ax.set_title("Shot Coordinate Spread by Venue-Season (size = shot count)")
    fig.tight_layout()
    plt.show()
else:
    print("No coordinate diagnostics available.")

## 7. Seasonal stability of venue bias

Is bias stable across seasons (persistent scorekeeper) or volatile (random noise, new scorekeeper, arena renovation)? For venues with multiple seasons of data, plot z-score trends.

In [ ]:
# Find venues that appear in 3+ seasons and have been flagged at least once
cur.execute("""
    SELECT venue_name, COUNT(DISTINCT season) AS n_seasons,
           SUM(bias_flag) AS times_flagged
    FROM venue_bias_diagnostics
    GROUP BY venue_name
    HAVING n_seasons >= 3
    ORDER BY times_flagged DESC
""")
multi_season_venues = cur.fetchall()

# Pick top venues by times flagged (or most data if none flagged)
MAX_VENUES_TO_PLOT = 8
venues_to_plot = [r[0] for r in multi_season_venues[:MAX_VENUES_TO_PLOT]]

if venues_to_plot:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

    for venue in venues_to_plot:
        cur.execute("""
            SELECT season, shot_count_z_score, distance_z_score
            FROM venue_bias_diagnostics
            WHERE venue_name = ?
            ORDER BY season
        """, (venue,))
        rows = cur.fetchall()
        if not rows:
            continue

        s_seasons = [str(r[0]) for r in rows]
        s_sc_z = [r[1] for r in rows]
        s_d_z = [r[2] for r in rows]

        ax1.plot(s_seasons, s_sc_z, marker="o", markersize=4, linewidth=1.2,
                 alpha=0.7, label=venue[:25])
        ax2.plot(s_seasons, s_d_z, marker="o", markersize=4, linewidth=1.2,
                 alpha=0.7, label=venue[:25])

    ax1.axhline(y=0, color="black", linewidth=0.5)
    ax1.axhline(y=2, color="#e74c3c", linestyle=":", alpha=0.5)
    ax1.axhline(y=-2, color="#e74c3c", linestyle=":", alpha=0.5)
    ax1.set_ylabel("Shot count z-score")
    ax1.set_title("Shot Count Z-Score Across Seasons")
    ax1.legend(fontsize=7, loc="upper left", ncol=2)

    ax2.axhline(y=0, color="black", linewidth=0.5)
    ax2.axhline(y=2, color="#e74c3c", linestyle=":", alpha=0.5)
    ax2.axhline(y=-2, color="#e74c3c", linestyle=":", alpha=0.5)
    ax2.set_ylabel("Distance z-score")
    ax2.set_xlabel("Season")
    ax2.set_title("Avg Distance Z-Score Across Seasons")
    ax2.legend(fontsize=7, loc="upper left", ncol=2)

    plt.xticks(rotation=45, fontsize=8)
    fig.tight_layout()
    plt.show()

    print("Stable lines near +/-2 = persistent bias (same scorekeeper pattern)")
    print("Volatile swings = noise or personnel changes")
else:
    print("Not enough multi-season venue data to analyze stability.")

## 8. Summary and recommendations

**Interpret the results above to decide:**

1. **Which venues need correction?** Venues with |z-score| > 2 that also show positive away-team asymmetry (section 5) are strong candidates for scorekeeper bias.

2. **Correction approach:**
   - If bias is small (z-score range < 0.5 across venues): venue as a categorical feature in the xG model may suffice.
   - If bias is moderate (z-score range 0.5-2.0): pre-correct shot counts by venue-season normalization factor.
   - If bias is large and stable: hierarchical partial pooling model may be needed.

3. **Seasonal stability:** If a venue's bias is stable across 3+ seasons, use pooled multi-season estimates. If volatile, use per-season correction only.

4. **Limitation:** The NHL API does not provide scorekeeper identity. Bias is venue-level only, which limits decomposition between scorekeeper change vs. arena renovation effects.

In [ ]:
conn.close()
print("Done.")